In [209]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec

import seaborn as sns

from pyfish import process_data, setup_figure, fish_plot
from matplotlib.sankey import Sankey

sns.set_style('ticks',
              rc={'axes.facecolor': (0, 0, 0, 0)})
sns.set_context('talk')

from matplotlib import rcParams, colors, cm
rcParams['font.family'] = 'sans-serif'
rcParams['figure.dpi'] = 150

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 100)

In [210]:
df = pd.read_excel("../../data/data.xlsx")

In [211]:
df["GPSC"] = df["GPSC"].astype("string")
df["serotype_cons"] = df["serotype_cons"].astype("string")
df["serotype_geno"] = df["serotype_geno"].astype("string")
df["serotype_pheno"] = df["serotype_pheno"].astype("string")
df["ST"] = df["ST"].astype("string")

In [212]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['year'] = df['date'].dt.year

In [213]:
df["year"].value_counts()

year
2018    95
2017    88
2024    82
2023    78
2019    77
2015    72
2005    72
2022    70
2016    63
2007    59
2004    59
2003    59
2014    52
2009    52
2006    52
2013    50
2008    47
2002    46
2021    44
2020    41
2010    36
2011    27
2012    25
Name: count, dtype: int64

In [214]:
# vaccines introduction
pcv7 = 2006
pcv13 = 2011
covid19 = 2020
pcv15 = 2023

In [215]:
# Define vaccine periods
def assign_vaccine_period(year):
    if year < 2005:
        return "pre-PCV7"
    elif 2005 <= year < 2011:
        return "PCV7"
    elif 2011 <= year < 2015:
        return "early-PCV13"
    elif 2015 <= year < 2020:
        return "late-PCV13"
    elif 2020 <= year < 2023:
        return "COVID19"
    else:  # 2020+ COVID impact
        return "PCV15"

df['vaccine_period'] = df['year'].apply(assign_vaccine_period)

prevalence = df.groupby(['vaccine_period', 'serotype_pheno']).size().reset_index(name='count')

# Get top 10 per period
top10_per_period = prevalence.groupby('vaccine_period', group_keys=False).apply(lambda x: x.nlargest(10, 'count'))

print(top10_per_period)

    vaccine_period serotype_pheno  count
21         COVID19              3     35
32         COVID19              8     19
16         COVID19            22F     15
14         COVID19            19F     12
33         COVID19             9N      9
13         COVID19            19A      7
18         COVID19            23B      7
0          COVID19            10A      5
2          COVID19            11A      3
4          COVID19             14      3
55           PCV15              3     25
66           PCV15              8     24
50           PCV15            22F     14
52           PCV15            23B      8
62           PCV15             38      8
67           PCV15             9N      8
36           PCV15            10A      7
43           PCV15            15B      6
51           PCV15            23A      6
42           PCV15            15A      5
93            PCV7              3     30
76            PCV7             14     27
99            PCV7              4     25
103           PC

/scratch/local/332170/ipykernel_3848849/894463791.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top10_per_period = prevalence.groupby('vaccine_period', group_keys=False).apply(lambda x: x.nlargest(10, 'count'))


In [216]:
# Define serotype groups by PCV category
pcv7 = ["4", "6B", "9V", "14", "18C", "19F", "23F"]
pcv13 = ["1", "3", "5", "6A", "7F", "19A"]
pcv15 = ["22F", "33F"]
pcv20 = ["8", "10A", "11A", "12F", "15B"]
ppv23 = ["2", "9N", "17F", "20"]
v116 = ["15A", "15C", "16F", "23A", "23B", "24F", "31", "35B"]

In [217]:
# Define serotype groups by PCV category
pcv7 = ["4", "6B", "9V", "14", "18C", "19F", "23F"]
pcv13 = ["1", "3", "5", "6A", "7F", "19A"]
pcv15 = ["22F", "33F"]
pcv20 = ["8", "10A", "11A", "12F", "15B"]
ppv23 = ["2", "9N", "17F", "20"]
v116 = ["15A", "15C", "16F", "23A", "23B", "24F", "31", "35B"]

VT = pcv7 + pcv13 + pcv15

df["isVT"] = df["serotype_cons"].isin(VT).astype(int)
df["isPCV7"] = df["serotype_cons"].isin(pcv7).astype(int)
df["isPCV13"] = df["serotype_cons"].isin(pcv13).astype(int)
df["isPCV15"] = df["serotype_cons"].isin(pcv15).astype(int)

In [218]:
df["age"] = df["age"].round().astype("Int64")

In [219]:
# Define age groups

def assign_age_groups(age):
    if pd.isna(age):
        return pd.NA
    elif age < 5:
        return "<5 years"
    elif age < 17:
        return "5-17 years"
    elif age < 45:
        return "18-44 years"
    elif age < 65:
        return "45-64 years"
    else:
        return ">=65 years"

df['age_group'] = df['age'].apply(assign_age_groups)

prevalence = df.groupby(['age_group', 'serotype_pheno']).size().reset_index(name='count')

In [220]:
prevalence

,age_group,serotype_pheno,count
0,18-44 years,0,3
1,18-44 years,1,4
2,18-44 years,10,1
3,18-44 years,10A,3
4,18-44 years,10B,1
...,...,...,...
201,>=65 years,7F,27
202,>=65 years,8,49
203,>=65 years,9,3
204,>=65 years,9N,30


In [221]:
df['isNVT'] = 1 - df['isVT']

In [222]:
vaccine_cols = ['pre-PCV7', 'PCV7', 'early-PCV13', 'late-PCV13', 'PCV15', 'COVID19']
for col in vaccine_cols:
    df[col] = (df['vaccine_period'] == col).astype(int)

In [223]:
counts = df["serotype_cons"].value_counts()

df["dominant_serotype"] = (
    df["serotype_cons"]
    .map(counts)
    .gt(24)
    .where(df["serotype_cons"].notna())
    .astype("Int64")
)

In [224]:
counts = df["GPSC"].value_counts()

df["dominant_GPSC"] = (
    df["GPSC"]
    .map(counts)
    .gt(24)
    .where(df["GPSC"].notna())
    .astype("Int64")
)

In [225]:
counts

GPSC
3               142
12               99
83               78
7                67
19               65
15               65
16               64
6                59
4                58
32               46
27               46
18               44
39               40
36               32
76               25
11               25
50               24
31               22
49               20
44               19
5                18
35               17
24               17
38               17
Not_assigned     14
57                9
29                9
10                8
1                 7
99                6
146               6
43                6
175               5
162               5
157               5
23                5
119               5
124               5
904;9             5
68                4
75                4
61                4
98                4
9                 4
46                4
67                4
48                3
13                3
115               3
2              

In [226]:
df_m = df[df.assembly == "missing"]

In [231]:
df_m

,lab-nr,working-nr,sample-nr,patient-nr,material,others,sepsis,HIV+,meningitis,covid,date,age,sex,serotype_pheno,serotype_geno,serogroup_match,serotype_match,serotype_cons,serotype_cons_reason,ST,GPSC,bad_quality,penicillin_men_geno,penicillin_nonmen_geno,penicillin_MIC_geno,pbp1a,pbp2b,pbp2x,ceftriaxone_men_geno,ceftriaxone_nonmen_geno,ceftriaxone_MIC_geno,amoxicillin_geno,amoxicillin_MIC_geno,CSP,ply_allele_aa,fasta,reads,sequencing_technology,reads_location,assembly,assembly_location,assembly_contigs,assembly_length,assembly_N50,assembly_closed,year,vaccine_period,isVT,isPCV7,isPCV13,isPCV15,age_group,isNVT,pre-PCV7,PCV7,early-PCV13,late-PCV13,PCV15,COVID19,dominant_serotype,dominant_GPSC
96,3152034,97,1197.25,421217.0,material_blood,1.0,0.0,0.0,0.0,0.0,2020-01-17,62,F,3,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,97,NaN,NaN,NaN,missing,NaN,NaN,NaN,NaN,not_assembled,2020,COVID19,0,0,0,0,45-64 years,1,0,0,0,0,0,1,<NA>,<NA>
123,3103882,124,1193.45,1421587.0,material_blood,1.0,0.0,0.0,0.0,0.0,2019-10-06,2,M,3,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,124,NaN,NaN,NaN,missing,NaN,NaN,NaN,NaN,not_assembled,2019,late-PCV13,0,0,0,0,<5 years,1,0,0,0,1,0,0,<NA>,<NA>
239,2831940,240,1176.51,701065.0,material_CSF,1.0,0.0,0.0,1.0,0.0,2018-03-03,8,M,35B,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,missing,NaN,NaN,NaN,NaN,not_assembled,2018,late-PCV13,0,0,0,0,5-17 years,1,0,0,0,1,0,0,<NA>,<NA>
261,2811262,262,1174.35,405337.0,material_other,1.0,0.0,0.0,0.0,0.0,2018-01-18,49,M,8,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A_262.fq.gz,PacBio,/storage/workspaces/ifik_miclab/ifik_miclab1/P...,missing,NaN,NaN,NaN,NaN,not_assembled,2018,late-PCV13,0,0,0,0,45-64 years,1,0,0,0,1,0,0,<NA>,<NA>
262,2809608,263,1174.14,457078.0,material_blood,1.0,0.0,0.0,0.0,0.0,2018-01-15,54,M,8,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A_263.fq.gz,PacBio,/storage/workspaces/ifik_miclab/ifik_miclab1/P...,missing,NaN,NaN,NaN,NaN,not_assembled,2018,late-PCV13,0,0,0,0,45-64 years,1,0,0,0,1,0,0,<NA>,<NA>
265,2809016,266,1173.81,1281646.0,material_blood,1.0,1.0,0.0,0.0,0.0,2018-01-13,55,F,8,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A_266.fq.gz,PacBio,/storage/workspaces/ifik_miclab/ifik_miclab1/P...,missing,NaN,NaN,NaN,NaN,not_assembled,2018,late-PCV13,0,0,0,0,45-64 years,1,0,0,0,1,0,0,<NA>,<NA>
267,2805700,268,1173.24,959397.0,material_blood,1.0,1.0,0.0,0.0,0.0,2018-01-06,3,M,8,<NA>,No_Match,No_Match,8,No genome annotation found,<NA>,<NA>,length,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A_268,A_268.fq.gz,PacBio,/storage/workspaces/ifik_miclab/ifik_miclab1/P...,missing,NaN,NaN,NaN,NaN,not_assembled,2018,late-PCV13,0,0,0,0,<5 years,1,0,0,0,1,0,0,1,<NA>
288,2789539,289,1170.76,1271133.0,material_other,1.0,0.0,0.0,0.0,0.0,2017-11-25,81,F,8,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A_289.fq.gz,PacBio,/storage/workspaces/ifik_miclab/ifik_miclab1/P...,missing,NaN,NaN,NaN,NaN,not_assembled,2017,late-PCV13,0,0,0,0,>=65 years,1,0,0,0,1,0,0,<NA>,<NA>
294,2771619,295,1169.70,1081734.0,material_blood,1.0,0.0,0.0,0.0,0.0,2017-10-13,39,F,8,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A_295.fq.gz,PacBio,/storage/workspaces/ifik_miclab/ifik_miclab1/P...,missing,NaN,NaN,NaN,NaN,not_assembled,2017,late-PCV13,0,0,0,0,18-44 years,1,0,0,0,1,0,0,<NA>,<NA>
306,2724683,307,1168.18,356414.0,material_blood,1.0,1.0,0.0,0.0,0.0,2017-06-16,66,M,8,<NA>,NaN,NaN,<NA>,NaN,<NA>,<NA>,missing,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A_307.fq.gz,PacBio,/storage/workspaces/ifik_miclab/ifik_miclab1/P...,missing,NaN,NaN,NaN,NaN,not_assembled,2017,late-PCV13,0,0,0,0,>

In [227]:
df_m.to_excel("../../data/data_missing.xlsx", index=None)

In [228]:
df_f = df[df.assembly == "present"]

In [229]:
df_f = df_f.drop(columns=[
    "CSP",
    "ply_allele_aa",
    "fasta",
    "reads",
    "sequencing_technology",
    "reads_location",
    "assembly",
    "assembly_location",
    "assembly_contigs",
    "assembly_length",
    "assembly_N50",
    "assembly_closed"
])

In [230]:
df_f.to_excel("../../data/data_cleaned.xlsx", index=None)